In [6]:
## Import the necessary packages 
import os
import pysam
import numpy as np
import pandas as pd
import math
import sys
import subprocess
import statsmodels.api as sm
import scipy
from scipy import stats
from scipy.stats import chi2
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import pysam
from typing import Optional

## Print out package versions
## Getting packages loaded into this notebook and their versions to allow for reproducibility
import pkg_resources
import types
from datetime import date

today = date.today()
date = today.strftime("%d-%b-%Y").upper()

## Define function 
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            name = val.__name__.split(".")[0]
        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages:
            name = poorly_named_packages[name]

        yield name

## Get a list of packages imported 
imports = list(set(get_imports()))

requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name != "pip":
        requirements.append((m.project_name, m.version))

## Print out packages and versions 
print(f"PACKAGE VERSIONS ({date})")
for r in requirements:
    print("\t{}=={}".format(*r))

## Also print which Python is being used
print("\nPYTHON INFO")
print(f"\tPython executable: {sys.executable}")

PACKAGE VERSIONS (03-DEC-2025)
	seaborn==0.13.2
	statsmodels==0.14.4
	matplotlib==3.7.2
	numpy==1.24.4
	pandas==2.0.3
	pysam==0.21.0
	scipy==1.11.1

PYTHON INFO
	Python executable: /usr/local/Anaconda/envs/py3.10/bin/python


# AA cis eQTL

## Pull out all gene associations with rs72546327 from AA cis-eQTLs

In [2]:
%%bash

zcat ${DATA_DIR}/Kachuri_2023/AA-cis-eQTL/12.allpairs.more.galasage.hg38.txt.gz | head -1  > ${DATA_DIR}/Kachuri_2023/AA_rs72546327.txt
zgrep rs72546327 ${DATA_DIR}/Kachuri_2023/AA-cis-eQTL/12.allpairs.more.galasage.hg38.txt.gz >> ${DATA_DIR}/Kachuri_2023/AA_rs72546327.txt

## Pull out all gene associations with LRRK2 from AA cis-eQTLs

In [3]:
%%bash

zcat ${DATA_DIR}/Kachuri_2023/AA-cis-eQTL/12.allpairs.more.galasage.hg38.txt.gz | head -1 > ${DATA_DIR}/Kachuri_2023/AA_all_LRRK2_results.txt
zgrep ENSG00000188906 ${DATA_DIR}/Kachuri_2023/AA-cis-eQTL/12.allpairs.more.galasage.hg38.txt.gz >> ${DATA_DIR}/Kachuri_2023/AA_all_LRRK2_results.txt

In [4]:
! head {DATA_DIR}/Kachuri_2023/AA_all_LRRK2_results.txt

gene_id	variant_id	tss_distance	ma_samples	ma_count	maf	pval_nominal	slope	slope_se	qval	pval_nominal_threshold	eQTL.flag	vid2
ENSG00000188906.16	rs116036419	465819	104	108	0.0713342	0.239052	0.0211602	0.0179572	1.53194e-22	0.000797038	FALSE	chr12_40662563_C_T_b38
ENSG00000188906.16	rs17489985	-49090	152	165	0.108983	0.050721	0.0285069	0.014565	1.53194e-22	0.000797038	FALSE	chr12_40147654_C_T_b38
ENSG00000188906.16	rs1822601	465333	469	596	0.393659	0.979642	-0.000236613	0.00926933	1.53194e-22	0.000797038	FALSE	chr12_40662077_T_A_b38
ENSG00000188906.16	rs73094449	-735559	163	172	0.113606	0.691222	0.00578637	0.014562	1.53194e-22	0.000797038	FALSE	chr12_39461185_T_C_b38
ENSG00000188906.16	rs4462419	465177	114	124	0.0819022	0.983758	0.000330472	0.0162278	1.53194e-22	0.000797038	FALSE	chr12_40661921_T_C_b38
ENSG00000188906.16	rs17442812	-47946	116	121	0.0799207	0.0106624	0.04408	0.0172155	1.53194e-22	0.000797038	FALSE	chr12_40148798_T_C_b38
ENSG00000188906.16	rs118148378	-48031	19	19	0.0125

# AFR cis eQTLs

## Pull out all gene associations with rs72546327 from AFR cis-eQTLs

In [20]:
%%bash

zcat ${DATA_DIR}/Kachuri_2023/AFRHp5-cis-eQTL/12.allpairs.more.galasage.hg38.txt.gz | head -1  > ${DATA_DIR}/Kachuri_2023/AFRHp5_rs72546327.txt
zgrep rs72546327 ${DATA_DIR}/Kachuri_2023/AFRHp5-cis-eQTL/12.allpairs.more.galasage.hg38.txt.gz >> ${DATA_DIR}/Kachuri_2023/AFRHp5_rs72546327.txt

## Pull out all gene associations with LRRK2 from AFR cis-eQTLs

In [3]:
%%bash

zcat ${DATA_DIR}/Kachuri_2023/AFRHp5-cis-eQTL/12.allpairs.more.galasage.hg38.txt.gz | head -1 > ${DATA_DIR}/Kachuri_2023/AFRHp5_all_LRRK2_results.txt
zgrep ENSG00000188906 ${DATA_DIR}/Kachuri_2023/AFRHp5-cis-eQTL/12.allpairs.more.galasage.hg38.txt.gz >> ${DATA_DIR}/Kachuri_2023/AFRHp5_all_LRRK2_results.txt

## Reformat
Keep +/- 1Mb in summary stats and eQTL data around LRRK2, and keep just rsID and pval for locusCompareR (run locally)

## Prep files for LocusCompare

In [20]:
aac = pd.read_csv(f"{DATA_DIR}/Kachuri_2023/AA_all_LRRK2_results.txt", sep="\t")
subset_aac = aac[['variant_id', 'pval_nominal']]
subset_aac.columns = ['rsid', 'pvalue']
subset_aac.to_csv(f"{DATA_DIR}/Kachuri_2023/forLocusCompare/AA_all_LRRK2_results.txt", sep="\t", index=False)

In [21]:
aac

,gene_id,variant_id,tss_distance,ma_samples,ma_count,maf,pval_nominal,slope,slope_se,qval,pval_nominal_threshold,eQTL.flag,vid2
0,ENSG00000188906.16,rs116036419,465819,104,108,0.071334,0.239052,0.021160,0.017957,1.531940e-22,0.000797,False,chr12_40662563_C_T_b38
1,ENSG00000188906.16,rs17489985,-49090,152,165,0.108983,0.050721,0.028507,0.014565,1.531940e-22,0.000797,False,chr12_40147654_C_T_b38
2,ENSG00000188906.16,rs1822601,465333,469,596,0.393659,0.979642,-0.000237,0.009269,1.531940e-22,0.000797,False,chr12_40662077_T_A_b38
3,ENSG00000188906.16,rs73094449,-735559,163,172,0.113606,0.691222,0.005786,0.014562,1.531940e-22,0.000797,False,chr12_39461185_T_C_b38
4,ENSG00000188906.16,rs4462419,465177,114,124,0.081902,0.983758,0.000330,0.016228,1.531940e-22,0.000797,False,chr12_40661921_T_C_b38
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11148,ENSG00000188906.16,rs12298792,243493,73,76,0.050198,0.454529,0.015728,0.021018,1.531940e-22,0.000797,False,chr12_40440237_C_T_b38
11149,ENSG00000188906.16,rs7980636,-560820,403,483,0.319022,0.037966,-0.020536,0.009877,1.531940e-22,0.000797,False,chr12_39635924_C_G_b38
11150,ENSG00000188906.16,rs11177347,393220,289,322,0.212682,0.919820,0.001136,0.011281,1.531940e-22,0.000797,False,chr12_40589964_T_C_b38
11151,ENSG00000188906.16,rs142116702,-547606,16,16,0.010568,0.276698,-0.050579,0.046462,1.531940e-22,0.000797,False,chr12_39649138_C_T_b38


In [22]:
afr = pd.read_csv(f"{DATA_DIR}/Kachuri_2023/AFRHp5_all_LRRK2_results.txt", sep="\t")
subset_afr = afr[['variant_id', 'pval_nominal']]
subset_afr.columns = ['rsid', 'pvalue']
subset_afr.to_csv(f"{DATA_DIR}/Kachuri_2023/forLocusCompare/AFRHp5_all_LRRK2_results.txt", sep="\t", index=False)

## Process GWAS summary statistics
* LRRK2 hg38 gnomAD: chr12:40196744-40369285

In [3]:
afr_gwas_path = f"{DATA_DIR}/fromLocusZoom/GP2_R11_AFR_fromLZ.txt.gz"
aac_meta_path = f"{DATA_DIR}/fromLocusZoom/AAC_Meta_fromLZ.txt.gz"
combined_meta_path = f"{DATA_DIR}/fromLocusZoom/AFR_AAC_fullMeta_fromLZ.txt.gz"

In [5]:
afr_gwas = pd.read_csv(afr_gwas_path, sep="\t")
aac_meta = pd.read_csv(aac_meta_path, sep="\t")
combined_meta = pd.read_csv(combined_meta_path, sep="\t")

In [13]:
def calculate_pvalue_and_subset(
    df: pd.DataFrame, 
    output_path: str,
    gene_chr: int = 12,
    gene_start: int = 40196744,
    gene_end: int = 40369285,
    buffer_mb: float = 1.0,
    compression: str = 'gzip',
    sep: str = '\t'
) -> pd.DataFrame:
    """
    Filters a genomic summary statistics DataFrame to a specific gene region 
    (+/- buffer), calculates the regular p-value, selects only 'rsid' and 'pvalue', 
    and saves the subset to file.
    """

    ONE_MB = 1_000_000
    buffer_bp = int(buffer_mb * ONE_MB)
    
    filter_start = gene_start - buffer_bp
    filter_end = gene_end + buffer_bp

    # Ensure the chromosome column is numeric
    df['chrom'] = pd.to_numeric(df['#chrom'], errors='coerce') 

    subset_df = df[
        (df['chrom'] == gene_chr) & 
        (df['pos'] >= filter_start) & 
        (df['pos'] <= filter_end)
    ].copy()

    # Apply the formula: P-value = 10^(-neg_log_pvalue)
    subset_df['pvalue'] = np.power(10, -subset_df['neg_log_pvalue'])
    
    final_df = subset_df[['rsid', 'pvalue']]

    final_df.to_csv(
        output_path, 
        sep=sep, 
        index=False
    )

    print(f"Subset shape (Chr {gene_chr}, {buffer_mb}Mb buffer): {final_df.shape}")
    print(f"Subset saved to: {output_path}")

    return final_df

In [ ]:
calculate_pvalue_and_subset(
    df=afr_gwas, 
    output_path=f"{DATA_DIR}/Kachuri_2023/forLocusCompare/AFR_GWAS_LRRK2_1Mb.tsv"
)

In [ ]:
calculate_pvalue_and_subset(
    df=afr_gwas, 
    output_path=f"{DATA_DIR}/Kachuri_2023/forLocusCompare/AFR_GWAS_LRRK2_1Mb.tsv"
)

In [ ]:
calculate_pvalue_and_subset(
    df=aac_meta, 
    output_path=f"{DATA_DIR}/Kachuri_2023/forLocusCompare/AAC_Meta_GWAS_LRRK2_1Mb.tsv"
)

In [ ]:
calculate_pvalue_and_subset(
    df=combined_meta, 
    output_path=f"{DATA_DIR}/Kachuri_2023/forLocusCompare/Combined_AFR_AAC_Meta_GWAS_LRRK2_1Mb.tsv"
)